<a href="https://colab.research.google.com/github/gehanimitali14feb-spec/RAG_based_ai_teching_assistance/blob/main/Rag_based_AI_teaching_Assisatnce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install -q openai-whisper
!unzip -q Audio.zip -d ./Audio

[Audio.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of Audio.zip or
        Audio.zip.zip, and cannot find Audio.zip.ZIP, period.


In [17]:
!apt-get install -y p7zip-full
!7z x -y -o./Audio Audio.zip

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
p7zip-full is already the newest version (16.02+transitional.1).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.

7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:2 OPEN_MAX:1048576

Scanning the drive for archives:
  0M Scan         1 file, 197218031 bytes (189 MiB)

Extracting archive: Audio.zip
 14% 1 Open           
ERRORS:
Headers Error
Unconfirmed start of archive


WARNINGS:
There are data after the end of archive

--
Path = Audio.zip
Type = zip
ERRORS:
Headers Error
Unconfirmed start of archive
WARNINGS:
There are data after the end of archive
Physical Size = 29294013
Tail Size = 167924018
Characteristics = Local

 97%    ERROR: Data Error : Audio/1_Installing VS Code & How Websites Work.mp3
100% - Audio/1_Installing VS Code & How Websites Work.mp3

In [18]:
import glob
import json
import os
import whisper

# Loads Whisper on Colab GPU
model = whisper.load_model("medium")

output_dir = "./Transcripts"
os.makedirs(output_dir, exist_ok=True)

# Recursively finds all audio files inside ./Audio or any subfolders
audio_extensions = ("*.mp3", "*.wav", "*.m4a")
audio_files = []
for ext in audio_extensions:
    audio_files.extend(glob.glob(f"./Audio/**/{ext}", recursive=True))

if not audio_files:
    print("No compatible audio files found. Make sure Audio.zip was extracted!")
else:
    print(f"Found {len(audio_files)} audio files. Starting transcription...\n")

    for audio_path in sorted(audio_files):
        file_name = os.path.basename(audio_path)
        print(f"Transcribing: {file_name}...")

        result = model.transcribe(
            audio=audio_path, language="hi", task="translate"
        )

        # Save formatted JSON with segment timestamps for RAG search
        json_name = os.path.splitext(file_name)[0] + ".json"
        json_path = os.path.join(output_dir, json_name)

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=4)

        print(f"✓ Saved: {json_name}\n")

    print("All transcriptions complete!")

Found 17 audio files. Starting transcription...

Transcribing: 12_Exercise 1 - Pure HTML Media Player.mp3...
✓ Saved: 12_Exercise 1 - Pure HTML Media Player.json

Transcribing: 13_Entities, Code tag and more on HTML.mp3...
✓ Saved: 13_Entities, Code tag and more on HTML.json

Transcribing: 14_Introduction to CSS.mp3...
✓ Saved: 14_Introduction to CSS.json

Transcribing: 15_Inline, Internal & External CSS.mp3...
✓ Saved: 15_Inline, Internal & External CSS.json

Transcribing: 16_Exercise 1 - Solution & Shoutouts.mp3...
✓ Saved: 16_Exercise 1 - Solution & Shoutouts.json

Transcribing: 17_CSS Selectors MasterClass.mp3...
✓ Saved: 17_CSS Selectors MasterClass.json

Transcribing: 18_CSS Box Model - Margin, Padding & Borders.mp3...
✓ Saved: 18_CSS Box Model - Margin, Padding & Borders.json

Transcribing: 2_Your First HTML Website.mp3...
✓ Saved: 2_Your First HTML Website.json

Transcribing: 3_Basic Structure of an HTML Website.mp3...
✓ Saved: 3_Basic Structure of an HTML Website.json

Transcr

In [19]:
import os
import zipfile

zip_path = 'Audio.zip'
extract_base = './Audio'
os.makedirs(extract_base, exist_ok=True)

print("Attempting resilient byte-by-byte file extraction...")

try:
    # Using zipfile in a way that doesn't rely entirely on the trailing central directory if corrupted
    with open(zip_path, 'rb') as f:
        with zipfile.ZipFile(f) as z:
            for member in z.infolist():
                if member.is_dir():
                    continue
                try:
                    # Re-map target location
                    clean_name = os.path.basename(member.filename)
                    target_path = os.path.join(extract_base, clean_name)

                    # Read bytes individually and save
                    print(f"Extracting: {clean_name}")
                    with z.open(member) as source, open(target_path, 'wb') as target:
                        target.write(source.read())
                except Exception as file_error:
                    print(f"Could not extract {member.filename}: {file_error}")
except Exception as e:
    print(f"Extraction failed: {e}")

print("\nChecking extracted files:")
print(os.listdir(extract_base))

Attempting resilient byte-by-byte file extraction...
Extracting: 1_Installing VS Code & How Websites Work.mp3
Could not extract Audio/1_Installing VS Code & How Websites Work.mp3: [Errno 22] Invalid argument
Extracting: 10_Video, Audio & Media in HTML.mp3
Could not extract Audio/10_Video, Audio & Media in HTML.mp3: [Errno 22] Invalid argument
Extracting: 11_Semantic Tags  in HTML.mp3
Could not extract Audio/11_Semantic Tags  in HTML.mp3: Bad magic number for file header
Extracting: 12_Exercise 1 - Pure HTML Media Player.mp3
Extracting: 13_Entities, Code tag and more on HTML.mp3
Extracting: 14_Introduction to CSS.mp3
Extracting: 15_Inline, Internal & External CSS.mp3
Extracting: 16_Exercise 1 - Solution & Shoutouts.mp3
Extracting: 17_CSS Selectors MasterClass.mp3
Extracting: 18_CSS Box Model - Margin, Padding & Borders.mp3
Extracting: 2_Your First HTML Website.mp3
Extracting: 3_Basic Structure of an HTML Website.mp3
Extracting: 4_Heading, Paragraphs and Links.mp3
Extracting: 5_Image, Li

In [20]:
import os
import zipfile

file_path = '/content/Audio.zip'

if os.path.exists(file_path):
    size_bytes = os.path.getsize(file_path)
    print(f"File size: {size_bytes / (1024 * 1024):.2f} MB ({size_bytes} bytes)")

    try:
        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            print("Zip check passed! Listing first 5 files:")
            print(zip_ref.namelist()[:5])
    except Exception as e:
        print(f"\nZip validation failed with error: {e}")
        print("\nPotential reasons:")
        if size_bytes == 0:
            print("- The file is completely empty (0 bytes).")
        else:
            print("- The file did not upload completely.")
            print("- The file is corrupted.")
            print("- The file is actually not a ZIP archive.")
else:
    print(f"Error: '{file_path}' does not exist in /content/ directory.")

File size: 188.08 MB (197218031 bytes)
Zip check passed! Listing first 5 files:
['Audio/1_Installing VS Code & How Websites Work.mp3', 'Audio/10_Video, Audio & Media in HTML.mp3', 'Audio/11_Semantic Tags  in HTML.mp3', 'Audio/12_Exercise 1 - Pure HTML Media Player.mp3', 'Audio/13_Entities, Code tag and more on HTML.mp3']


In [22]:
import glob
import json
import os

def load_all_transcripts(transcripts_dir="./Transcripts"):
    segments_db = []
    for fpath in sorted(glob.glob(os.path.join(transcripts_dir, "*.json"))):
        lec_name = os.path.splitext(os.path.basename(fpath))[0]
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)
            for seg in data.get("segments", []):
                s_min, s_sec = int(seg["start"] // 60), int(seg["start"] % 60)
                e_min, e_sec = int(seg["end"] // 60), int(seg["end"] % 60)
                segments_db.append({
                    "lecture": lec_name,
                    "timestamp": f"{s_min:02d}:{s_sec:02d} - {e_min:02d}:{e_sec:02d}",
                    "text": seg.get("text", "").strip()
                })
    return segments_db

DB = load_all_transcripts()
print(f"Loaded {len(DB)} timestamped segments into memory.")

def ask_rag(question):
    words = [w.lower() for w in question.split() if len(w) > 3]
    matches = []
    for item in DB:
        score = sum(1 for w in words if w in item["text"].lower() or w in item["lecture"].lower())
        if score > 0:
            matches.append((score, item))

    matches.sort(key=lambda x: x[0], reverse=True)
    top_matches = matches[:5]

    if not top_matches:
        print(f"No relevant lectures found for: '{question}'")
        return

    print(f"\nAnswer for: '{question}'\n" + "="*50)
    for _, item in top_matches:
        print(f"📌 Lecture: {item['lecture']}")
        print(f"   Timestamp: {item['timestamp']}")
        print(f"   Context:   \"{item['text']}\"\n")

Loaded 5319 timestamped segments into memory.


In [23]:
ask_rag("When was CSS box model taught?")


Answer for: 'When was CSS box model taught?'
📌 Lecture: 18_CSS Box Model - Margin, Padding & Borders
   Timestamp: 02:39 - 02:41
   Context:   "which you will know when I will add a background"

📌 Lecture: 18_CSS Box Model - Margin, Padding & Borders
   Timestamp: 10:59 - 11:01
   Context:   "whenever we set a height"

📌 Lecture: 18_CSS Box Model - Margin, Padding & Borders
   Timestamp: 11:05 - 11:07
   Context:   "when I say its height is 200"

📌 Lecture: 18_CSS Box Model - Margin, Padding & Borders
   Timestamp: 12:05 - 12:07
   Context:   "when I am saying box sizing and border box"

📌 Lecture: 18_CSS Box Model - Margin, Padding & Borders
   Timestamp: 16:31 - 16:33
   Context:   "when you have to apply box sizing border box"



In [24]:
ask_rag("Where do we learn about media player in HTML?")


Answer for: 'Where do we learn about media player in HTML?'
📌 Lecture: 12_Exercise 1 - Pure HTML Media Player
   Timestamp: 00:00 - 00:06
   Context:   "Guys, in today's video, I will give you an exercise and I will also give shout out to all the people who will solve this exercise."

📌 Lecture: 12_Exercise 1 - Pure HTML Media Player
   Timestamp: 00:06 - 00:09
   Context:   "So, write in the comment section, challenge accepted."

📌 Lecture: 12_Exercise 1 - Pure HTML Media Player
   Timestamp: 00:09 - 00:15
   Context:   "If you accept my challenge, I will give you a question and you have to solve it."

📌 Lecture: 12_Exercise 1 - Pure HTML Media Player
   Timestamp: 00:15 - 00:21
   Context:   "And it will be based on the concepts we have studied in the Sigma Web Development course."

📌 Lecture: 12_Exercise 1 - Pure HTML Media Player
   Timestamp: 00:21 - 00:24
   Context:   "So, I am very excited to give you a question."

